Pivot the DataFrame to get a wide format: dates as index, tickers as columns with close prices.

Use that pivoted DataFrame to calculate log returns.

Fit a GARCH(1,1) model for each ticker’s returns.

Forecast 1-day-ahead volatility per ticker.

Calculate correlation matrix from historical returns.

Combine volatilities and correlations into portfolio volatility using weights.

In [6]:
pip install arch

Note: you may need to restart the kernel to use updated packages.


In [7]:
# import pandas as pd
# import numpy as np
# from arch import arch_model

# def forecast_portfolio_volatility(raw_df: pd.DataFrame, weights: dict):
#     # Step 1: Pivot the long format to wide format for close prices
#     price_df = raw_df.pivot(index='date', columns='ticker', values='close')
#     price_df.index = pd.to_datetime(price_df.index)
#     price_df = price_df.sort_index()

#     # Step 2: Calculate log returns
#     log_returns = np.log(price_df / price_df.shift(1)).dropna()

#     # Step 3: Filter to only tickers that exist in weights
#     tickers = [t for t in price_df.columns if t in weights]
#     log_returns = log_returns[tickers]

#     # Step 4: Normalize weights
#     total_weight = sum(weights[t] for t in tickers)
#     weight_vector = np.array([weights[t] / total_weight for t in tickers])

#     # Step 5: GARCH forecast for each ticker
#     forecasted_vols = []
#     for ticker in tickers:
#         returns = log_returns[ticker] * 100  # GARCH prefers percentage scale
#         model = arch_model(returns, vol='GARCH', p=1, q=1)
#         res = model.fit(disp='off')
#         forecast = res.forecast(horizon=1)
#         sigma = np.sqrt(forecast.variance.values[-1][0]) / 100  # back to raw scale
#         forecasted_vols.append(sigma)

#     # Step 6: Correlation matrix from historical returns
#     correlation_matrix = log_returns.corr().values

#     # Step 7: Construct forecasted covariance matrix
#     forecasted_vol_matrix = np.outer(forecasted_vols, forecasted_vols)
#     forecasted_cov_matrix = forecasted_vol_matrix * correlation_matrix

#     # Step 8: Calculate portfolio volatility
#     portfolio_variance = weight_vector.T @ forecasted_cov_matrix @ weight_vector
#     portfolio_volatility = np.sqrt(portfolio_variance)

#     return {
#         "portfolio_volatility": portfolio_volatility,
#         "individual_vols": dict(zip(tickers, forecasted_vols)),
#         "correlation_matrix": pd.DataFrame(correlation_matrix, index=tickers, columns=tickers),
#         "log_returns": log_returns
#     }


In [8]:
import pandas as pd
import numpy as np
from arch import arch_model

def forecast_portfolio_volatility(raw_df: pd.DataFrame, weights: dict):
    # Step 1: Prepare data
    raw_df['Date'] = pd.to_datetime(raw_df['Date'])  # Ensure datetime
    raw_df = raw_df.sort_values('Date')              # Sort by date
    raw_df = raw_df.set_index('Date')                # Set date as index

    # Step 2: Filter columns by tickers in weights
    tickers = [t for t in weights.keys() if t in raw_df.columns]
    price_df = raw_df[tickers].copy()

    # Step 3: Calculate log returns
    log_returns = np.log(price_df / price_df.shift(1)).dropna()

    # Step 4: Normalize weights
    total_weight = sum(weights[t] for t in tickers)
    weight_vector = np.array([weights[t] / total_weight for t in tickers])

    # Step 5: GARCH forecast for each ticker
    forecasted_vols = []
    for ticker in tickers:
        returns = log_returns[ticker] * 100  # GARCH prefers percentage scale
        model = arch_model(returns, vol='GARCH', p=2, q=3)
        res = model.fit(disp='off')
        forecast = res.forecast(horizon=1)
        sigma = np.sqrt(forecast.variance.values[-1][0]) / 100  # back to raw scale
        forecasted_vols.append(sigma)

    # Step 6: Correlation matrix from historical returns
    correlation_matrix = log_returns.corr().values

    # Step 7: Construct forecasted covariance matrix
    forecasted_vol_matrix = np.outer(forecasted_vols, forecasted_vols)
    forecasted_cov_matrix = forecasted_vol_matrix * correlation_matrix

    # Step 8: Calculate portfolio volatility
    portfolio_variance = weight_vector.T @ forecasted_cov_matrix @ weight_vector
    portfolio_volatility = np.sqrt(portfolio_variance)

    return {
        "portfolio_volatility": portfolio_volatility,
        "individual_vols": dict(zip(tickers, forecasted_vols)),
        "correlation_matrix": pd.DataFrame(correlation_matrix, index=tickers, columns=tickers),
        "log_returns": log_returns
    }


In [10]:
# Example: Load or define your historical price data
# df_prices = pd.read_csv("your_price_data.csv")  # must have columns: ['date', 'ticker', 'close']

# Your weights
weights = {
    "AAPL": 0.5,
    "GOOG": 0.3,
    "MSFT": 0.2
}
df_prices=pd.read_csv("./PCA_Actual_Prices1.csv")


df_prices


,Date,A,AAP,AAPL,ABT,ACN,ADBE,ADI,ADM,ADP,...,WY,WYNN,XEL,XOM,XRAY,XRX,YUM,ZBH,ZBRA,ZION
0,2006-04-07,23.148590,35.310947,2.143868,13.260740,21.095428,36.099998,24.805746,23.281313,23.413929,...,14.436615,42.614071,9.126107,32.493980,23.888378,18.261936,12.051119,57.900055,43.770000,58.285786
1,2006-04-10,22.763783,35.250778,2.100232,13.019472,20.829950,36.950001,24.325201,22.732697,23.253765,...,14.242338,41.604065,9.011406,32.096241,24.453604,17.993372,11.986113,57.274765,43.660000,57.613182
2,2006-04-11,22.198597,34.881256,2.066528,13.047309,20.815592,37.500000,24.084915,22.827276,23.093586,...,14.109606,41.263817,9.011406,32.415466,24.411100,17.981176,11.938602,57.465824,43.500000,57.683266
3,2006-04-12,22.228664,33.867214,2.046063,12.911805,20.629032,37.279999,23.857273,23.047989,23.003519,...,13.884547,40.610287,8.991465,32.446869,24.470610,17.846893,11.954898,57.248718,42.860001,56.989632
4,2006-04-13,21.843849,34.219551,2.007544,12.970989,21.088259,36.970001,24.009041,23.577681,23.023523,...,13.859534,41.388023,8.921641,32.164261,24.419607,17.639376,12.007530,56.875278,42.400002,56.688362
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3317,2019-06-13,67.347359,135.645889,46.696117,74.139366,170.780396,276.839996,93.922745,34.595238,145.605881,...,20.147099,109.385002,49.688438,56.187469,52.137367,24.711082,98.013863,111.840508,193.630005,36.269424
3318,2019-06-14,67.356934,135.709000,46.686489,74.446960,171.113480,276.299988,95.191589,34.841621,145.623596,...,20.323971,111.739723,49.970013,56.680801,52.201515,24.895121,98.049873,111.006485,194.000000,36.729256
3319,2019-06-17,67.040749,136.781204,46.347439,74.383644,171.131973,274.279999,93.607765,34.654709,146.588821,...,20.669668,110.214394,50.457527,56.430347,52.018250,24.364235,98.778603,108.838058,190.809998,36.852432
3320,2019-06-18,67.347359,136.727188,46.623974,74.229843,170.142075,275.600006,94.579666,34.612236,146.943054,...,20.637510,110.281136,50.331448,56.991982,51.917454,24.265135,98.211784,109.097527,192.990005,36.162682


In [11]:
df_prices['Date'] = pd.to_datetime(df_prices['Date'])

# Get tickers from weights dict
tickers = list(weights.keys())

# Keep only 'date' and selected ticker columns
df_filtered = df_prices[['Date'] + tickers].copy()

In [12]:
df_filtered

,Date,AAPL,GOOG,MSFT
0,2006-04-07,2.143868,10.192837,19.385588
1,2006-04-10,2.100232,10.068394,19.167538
2,2006-04-11,2.066528,10.321741,19.195677
3,2006-04-12,2.046063,10.155157,19.083130
4,2006-04-13,2.007544,10.137556,19.132370
...,...,...,...,...
3317,2019-06-13,46.696117,53.597641,124.675781
3318,2019-06-14,46.686489,54.181870,125.462784
3319,2019-06-17,46.347439,54.011673,125.586029
3320,2019-06-18,46.623974,54.367489,125.965294


In [13]:


result = forecast_portfolio_volatility(df_filtered, weights)

print("Predicted Portfolio Volatility:", result["portfolio_volatility"])
print("Individual Forecasted Volatilities:", result["individual_vols"])
print("Correlation Matrix:\n", result["correlation_matrix"])

Predicted Portfolio Volatility: 0.013883657288384735
Individual Forecasted Volatilities: {'AAPL': np.float64(0.016887743495626092), 'GOOG': np.float64(0.01711347367859217), 'MSFT': np.float64(0.014934382364054588)}
Correlation Matrix:
           AAPL      GOOG     MSFT
AAPL  1.000000  0.529835  0.46723
GOOG  0.529835  1.000000  0.54358
MSFT  0.467230  0.543580  1.00000
